# Perceptronul și gradient descent

Construim un neuron artificial fără biblioteci de machine learning. Vom calcula predicția, pierderea și gradientele, apoi vom antrena modelul pas cu pas pe problema porții logice OR.

## Ce trebuie să știi

Lecția presupune funcții Python, liste și noțiuni elementare despre derivate. Ideea centrală este regula lanțului: dacă $L$ depinde de $z$, iar $z$ depinde de $w$, atunci

$$\frac{\partial L}{\partial w}=\frac{\partial L}{\partial z}\frac{\partial z}{\partial w}. $$

## 1. Setul de date

Poarta OR produce `1` dacă cel puțin una dintre intrări este `1`. Cele patru combinații posibile formează întregul nostru set de antrenare.

In [1]:
X = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
y = [0.0, 1.0, 1.0, 1.0]
list(zip(X, y))

[((0.0, 0.0), 0.0),
 ((0.0, 1.0), 1.0),
 ((1.0, 0.0), 1.0),
 ((1.0, 1.0), 1.0)]

## 2. Propagarea înainte

Neuronul calculează mai întâi combinația liniară

$$z=w_1x_1+w_2x_2+b,$$

apoi aplică funcția sigmoidă $\sigma(z)=1/(1+e^{-z})$. Rezultatul poate fi interpretat ca o probabilitate între 0 și 1.

In [2]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def forward(x, w, b):
    z = sum(xi * wi for xi, wi in zip(x, w)) + b
    return sigmoid(z)

w = [0.0, 0.0]
b = 0.0
print("Predicție inițială:", forward((1.0, 0.0), w, b))

Predicție inițială: 0.5


## 3. Funcția de pierdere

Binary cross-entropy penalizează o probabilitate atribuită greșit:

$$L(y,\hat y)=-[y\log(\hat y)+(1-y)\log(1-\hat y)].$$

Adăugăm un epsilon pentru a evita evaluarea lui $\log(0)$.

In [3]:
def binary_cross_entropy(y_real, y_pred):
    eps = 1e-12
    p = min(max(y_pred, eps), 1 - eps)
    return -(y_real * math.log(p) + (1 - y_real) * math.log(1 - p))

print("Predicție 0.9, etichetă 1:", round(binary_cross_entropy(1, 0.9), 4))
print("Predicție 0.1, etichetă 1:", round(binary_cross_entropy(1, 0.1), 4))

Predicție 0.9, etichetă 1: 0.1054
Predicție 0.1, etichetă 1: 2.3026


## 4. Gradientele

Pentru combinația sigmoidă + binary cross-entropy, derivata pierderii față de $z$ se simplifică la $\hat y-y$. De aici rezultă:

$$\frac{\partial L}{\partial w_j}=(\hat y-y)x_j, \qquad \frac{\partial L}{\partial b}=\hat y-y.$$

Gradientul indică direcția de creștere maximă. Pentru a reduce pierderea, actualizăm parametrii în direcția opusă.

In [4]:
def gradient_exemplu(x, y_real, w, b):
    y_pred = forward(x, w, b)
    eroare = y_pred - y_real
    grad_w = [eroare * xi for xi in x]
    grad_b = eroare
    return grad_w, grad_b

gradient_exemplu((1.0, 0.0), 1.0, w, b)

([-0.5, -0.0], -0.5)

## 5. Bucla de antrenare

În fiecare epocă acumulăm gradientele pentru toate exemplele, calculăm media și actualizăm parametrii. Acesta este batch gradient descent. Rata de învățare `lr` controlează mărimea pasului.

In [5]:
w = [0.0, 0.0]
b = 0.0
lr = 0.5

for epoca in range(1, 601):
    grad_w = [0.0, 0.0]
    grad_b = 0.0
    pierdere = 0.0

    for x, y_real in zip(X, y):
        y_pred = forward(x, w, b)
        pierdere += binary_cross_entropy(y_real, y_pred)
        eroare = y_pred - y_real
        for j in range(2):
            grad_w[j] += eroare * x[j]
        grad_b += eroare

    n = len(X)
    for j in range(2):
        w[j] -= lr * grad_w[j] / n
    b -= lr * grad_b / n

    if epoca == 1 or epoca % 200 == 0:
        print(f"epoca {epoca:4d} | loss {pierdere / n:.4f}")

print("Parametri:", [round(v, 2) for v in w], round(b, 2))

epoca    1 | loss 0.6931
epoca  200 | loss 0.1521
epoca  400 | loss 0.0859
epoca  600 | loss 0.0589
Parametri: [5.73, 5.73] -2.26


## 6. Evaluarea modelului

Transformăm probabilitatea în clasă folosind pragul 0.5. Este important să inspectăm și probabilitățile: ele arată cât de departe se află fiecare exemplu de frontiera de decizie.

In [6]:
for x in X:
    probabilitate = forward(x, w, b)
    clasa = int(probabilitate >= 0.5)
    print(f"{x} → p={probabilitate:.3f}, clasa={clasa}")

(0.0, 0.0) → p=0.094, clasa=0
(0.0, 1.0) → p=0.970, clasa=1
(1.0, 0.0) → p=0.970, clasa=1
(1.0, 1.0) → p=1.000, clasa=1


## 7. Limita unui singur neuron

Perceptronul trasează o frontieră liniară. Poarta XOR nu este liniar separabilă, deci niciun set de parametri $w_1,w_2,b$ nu poate clasifica perfect toate cele patru puncte. Un strat ascuns permite combinarea mai multor frontiere și rezolvă această limitare.

Aceasta este motivația esențială pentru rețelele cu mai multe straturi — nu doar mai mulți parametri, ci reprezentări mai expresive.

## Proiect de laborator

1. Modifică etichetele pentru porțile AND și NAND și compară parametrii învățați.
2. Înregistrează pierderea la fiecare epocă și construiește graficul curbei de învățare.
3. Compară ratele `0.01`, `0.1`, `0.5` și `2.0`. Explică diferențele.
4. Verifică numeric gradientul lui `w[0]` folosind aproximarea cu diferențe finite:

$$\frac{\partial L}{\partial w}\approx\frac{L(w+\varepsilon)-L(w-\varepsilon)}{2\varepsilon}. $$

**Provocare:** adaugă un strat ascuns cu doi neuroni și antrenează o rețea pentru XOR.